In [0]:
#WRITE A PYSPARK CODE TO CALCULATE THE PERCENTAGE DIFFERENCE BETWEEN TOTAL SALES IN QUARTER 1 AND QUARTER 2 OF EACH YEAR FROM THE SALES TABLE

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, quarter, sum, round

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("QuarterlySalesDifference") \
    .getOrCreate()

# Sample data: sales data with date and amount
data = [
    ("2021-01-15", 5000),  # Q1
    ("2021-02-20", 7000),  # Q1
    ("2021-04-10", 6000),  # Q2
    ("2021-05-15", 8000),  # Q2
    ("2022-01-10", 5500),  # Q1
    ("2022-02-25", 7500),  # Q1
    ("2022-04-05", 6500),  # Q2
    ("2022-06-12", 9000),  # Q2
]

# Define schema and create DataFrame
columns = ["date", "sales_amount"]
df = spark.createDataFrame(data, columns)

# Convert 'date' to DateType
df = df.withColumn("date", col("date").cast("date"))

# Extract year and quarter from the 'date' column
df = df.withColumn("year", year("date")) \
       .withColumn("quarter", quarter("date"))

# Calculate the total sales for each quarter (Q1 and Q2)
quarter_sales = df.groupBy("year", "quarter") \
                  .agg(sum("sales_amount").alias("total_sales"))

# Filter out only Q1 and Q2 sales
q1_q2_sales = quarter_sales.filter(col("quarter").isin(1, 2))

# Pivot the data to have Q1 and Q2 sales in separate columns
pivot_sales = q1_q2_sales.groupBy("year") \
                         .pivot("quarter") \
                         .agg(sum("total_sales"))

# Rename columns to make them more readable
pivot_sales = pivot_sales.withColumnRenamed("1", "Q1_sales") \
                         .withColumnRenamed("2", "Q2_sales")

# Calculate the percentage difference between Q1 and Q2 sales
result = pivot_sales.withColumn(
    "percentage_difference",
    round(((col("Q2_sales") - col("Q1_sales")) / col("Q1_sales")) * 100, 2)
)

# Show the result
result.show()


+----+--------+--------+---------------------+
|year|Q1_sales|Q2_sales|percentage_difference|
+----+--------+--------+---------------------+
|2022|   13000|   15500|                19.23|
|2021|   12000|   14000|                16.67|
+----+--------+--------+---------------------+

